In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.5 MB/s eta 0:00:00


In [4]:
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from clearml import Task

In [ ]:
task = Task.init(
    project_name="cross-lingual-lm",
    task_name="swahili_continued_pretraining_roberta_low_resource",
    task_type=Task.TaskTypes.training
)

logger = task.get_logger()

set_seed(42)

ClearML Task: created new task id=a2b28dd0a32644238dd148771f0d9af5


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


2026-04-19 18:23:20,309 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/9bf855ce67034d8d9b44bf6ae4fbf457/experiments/a2b28dd0a32644238dd148771f0d9af5/output/log


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/english_lm_roberta"

MAX_LENGTH = 128
BATCH_SIZE = 8
GRAD_ACCUM = 4

EPOCHS = 3
LR = 3e-5                   # lower LR to preserve EN knowledge

SAMPLE_SIZE = 10000

OUTPUT_DIR = "./models/swahili_continued"

task.connect({
    "model_path": MODEL_PATH,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "sample_size": SAMPLE_SIZE
})


{'model_path': '/content/drive/MyDrive/nlp_project/models/english_lm_roberta',
 'max_length': 128,
 'batch_size': 8,
 'grad_accum': 4,
 'epochs': 3,
 'learning_rate': 3e-05,
 'sample_size': 10000}

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ngusadeep/Swahili-Corpus-Dataset")["train"]

# Remove empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

# Limit dataset size (low-resource simulation)
dataset_dict = dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)

raw_train_data = dataset_dict["train"]
eval_data = dataset_dict["test"]

train_data = raw_train_data.select(range(min(SAMPLE_SIZE, len(raw_train_data))))
eval_data = eval_data.select(range(min(3000, len(eval_data))))

print(f"Train size: {len(train_data)}")
print(f"Eval size:  {len(eval_data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning:


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).



README.md: 0.00B [00:00, ?B/s]

Swahili_Corpus_combined.txt:   0%|          | 0.00/253M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Train size: 10000
Eval size:  3000


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = train_data.map(
    tokenize,
    batched=True,
    num_proc=os.cpu_count(),
    remove_columns=["text"]
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Map (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
import torch

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,
    learning_rate=LR,

    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,

    fp16=True,

    dataloader_num_workers=4,

    report_to=["clearml"],
    run_name="roberta_swahili_low_resource",

    optim="adamw_torch",
    lr_scheduler_type="linear",
    warmup_ratio=0.1
)
torch.manual_seed(42)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

logger.report_text("swahili continued pretraining (low-resource) finished")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning:

This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.



2026-04-19 18:25:05,453 - clearml.Task - WARNING - Parameters must be of builtin type (Transformers/accelerator_config[AcceleratorConfig])


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning:

This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.



Step,Training Loss
100,16.386741
200,14.211233
300,13.015480
400,12.341791
500,11.710210
600,11.596553
700,11.246606
800,11.102412
900,10.896953


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-19 18:32:16,417 - clearml.frameworks - INFO - Found existing registered model id=ed3931a0a365422ab1b26b873033fcc6 [/content/models/swahili_continued/checkpoint-939/training_args.bin] reusing it.
2026-04-19 18:32:23,599 - clearml.frameworks - INFO - Found existing registered model id=60d6f4bd3ecc42e59682231ab6f50082 [/content/models/swahili_continued/checkpoint-939/optimizer.pt] reusing it.
2026-04-19 18:32:26,784 - clearml.frameworks - INFO - Found existing registered model id=ade9640e8d4640978ca4c09c05e09899 [/content/models/swahili_continued/checkpoint-939/scheduler.pt] reusing it.
2026-04-19 18:32:30,015 - clearml.frameworks - INFO - Found existing registered model id=088246652dbb4a0caa8b8d451f71e1ac [/content/models/swahili_continued/checkpoint-939/scaler.pt] reusing it.
2026-04-19 18:32:33,769 - clearml.frameworks - INFO - Found existing registered model id=be3de7dffdfc4ff78ec5fb4046074fc5 [/content/models/swahili_continued/checkpoint-939/rng_state.pth] reusing it.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-19 18:32:45,732 - clearml.frameworks - INFO - Found existing registered model id=9651d032d8724612b50b4089c3cb1952 [/content/models/swahili_continued/training_args.bin] reusing it.
swahili continued pretraining (low-resource) finished


In [ ]:
import shutil
import os

shutil.copytree(OUTPUT_DIR, '/content/drive/MyDrive/nlp_project/models/roberta_swahili', dirs_exist_ok=True)

'/content/drive/MyDrive/nlp_project/models/roberta_swahili'

# Evaluate on EN PPL

In [ ]:
import math
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import math
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [ ]:
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"

MAX_LENGTH = 128
BATCH_SIZE = 8
EVAL_SAMPLES = 3000

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

dataset = dataset.filter(lambda x: x["text"] and len(x["text"].strip()) > 0)
dataset = dataset.select(range(min(EVAL_SAMPLES, len(dataset))))

dataset

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2891
})

In [ ]:
def eval_ppl(MODEL_PATH, dataset):
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
    model.to(device)
    model.eval()

    tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    args = TrainingArguments(
        output_dir="./eval_en_tmp",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=args,
        eval_dataset=tokenized_dataset,
        data_collator=collator
    )


    torch.manual_seed(42)
    metrics = trainer.evaluate()

    loss = metrics["eval_loss"]
    perplexity = math.exp(loss)

    print(f"Loss: {loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")

In [ ]:
eval_ppl("/content/drive/MyDrive/nlp_project/models/roberta_swahili", dataset)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

Loss: 1.8187
Perplexity: 6.16


# Evaluate on Swahili PPL

In [ ]:
dataset = load_dataset("ngusadeep/Swahili-Corpus-Dataset")["train"]

dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

dataset_dict = dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)

raw_train_data = dataset_dict["train"]
eval_data = dataset_dict["test"]

eval_data = eval_data.select(range(min(3000, len(eval_data))))

print(f"Eval size:  {len(eval_data)}")

Eval size:  3000


In [ ]:
eval_ppl("/content/drive/MyDrive/nlp_project/models/roberta_swahili", eval_data)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loss: 2.5973
Perplexity: 13.43


In [7]:
MODEL_PATH = '/content/drive/MyDrive/nlp_project/models/roberta_swahili'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
model.eval()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [8]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)

examples = [
    f"Habari za {tokenizer.mask_token}?",
    f"Mimi ni {tokenizer.mask_token}.",
    f"Jina langu ni {tokenizer.mask_token}."
]

for ex in examples:
    print(f"\nPrompt: {ex}")
    for res in fill_mask(ex):
        print(f"  {res['score']:.4f} -> {res['token_str']}")


Prompt: Habari za <mask>?
  0.0948 ->  ya
  0.0516 ->  
  0.0386 ->  ba
  0.0257 ->  tu
  0.0232 ->  wa

Prompt: Mimi ni <mask>.
  0.0629 ->  
  0.0610 ->  m
  0.0378 ->  k
  0.0296 ->  ya
  0.0271 ->  ni

Prompt: Jina langu ni <mask>.
  0.0649 ->  m
  0.0608 ->  
  0.0563 ->  k
  0.0471 ->  ni
  0.0360 ->  n
